In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#学習データ(train.csv)とテストデータ(test.csv)のデータフレームへの読み込み
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
test_ids = test['Id']  # 提出用に保存

In [ ]:
pip install japanize-matplotlib

In [ ]:
# インポート
import pandas as pd               # データ操作・分析用（表形式のデータ処理に便利）
import matplotlib.pyplot as plt   # 基本的なグラフ描画ライブラリ（折れ線・棒グラフなど）
import seaborn as sns             # 高機能なグラフ描画ライブラリ（ヒートマップや分布図など）
import math                       # 数学関数ライブラリ（対数・平方根などの計算に使用）
import japanize_matplotlib        # matplotlibで日本語を正しく表示するためのライブラリ
from sklearn.model_selection import train_test_split  # データの分割（学習用とテスト用）に使用
from sklearn.linear_model import LinearRegression      # 線形回帰モデル（連続値の予測に使用）
from sklearn.preprocessing import StandardScaler

In [ ]:
# 欠損値がある列とその欠損数を表示（学習データ）
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)
print("【学習データの欠損値】")
print(missing_train)

# 欠損値がある列とその欠損数を表示（テストデータ）
missing_test = test.isnull().sum()
missing_test = missing_test[missing_test > 0].sort_values(ascending=False)
print("\n【テストデータの欠損値】")
print(missing_test)

In [ ]:
# 数値の特徴量を抽出
numeric_train = train.select_dtypes(include=['number']).columns
numeric_test = test.select_dtypes(include=['number']).columns

# 学習データ：欠損値を各列の平均で補完
for col in numeric_train:
    if train[col].isnull().any():
        train[col].fillna(train[col].mean(), inplace=True)

# テストデータ：欠損値を各列の平均で補完
for col in numeric_test:
    if test[col].isnull().any():
        test[col].fillna(test[col].mean(), inplace=True)

In [ ]:
# 数値でない特徴量を抽出し、欠損値を 'Missing' に置き換え
non_numeric_train = train.select_dtypes(exclude=['number']).columns
train[non_numeric_train] = train[non_numeric_train].fillna('Missing')

# 数値でない特徴量を抽出し、欠損値を 'Missing' に置き換え
non_numeric_test = test.select_dtypes(exclude=['number']).columns
test[non_numeric_test] = test[non_numeric_test].fillna('Missing')

In [ ]:
# 欠損値がある列とその欠損数を表示（学習データ）
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)
print("【学習データの欠損値】")
print(missing_train)

# 欠損値がある列とその欠損数を表示（テストデータ）
missing_test = test.isnull().sum()
missing_test = missing_test[missing_test > 0].sort_values(ascending=False)
print("\n【テストデータの欠損値】")
print(missing_test)

In [ ]:
# 学習データから数値型の列のみ抽出
numeric_train = train.select_dtypes(include=['int64', 'float64'])
# テストデータから数値型の列のみ抽出
numeric_test = test.select_dtypes(include=['int64', 'float64'])

In [ ]:
# 学習データ：SalePriceとの相関係数を計算
correlations = numeric_train.corr()['SalePrice'].sort_values(ascending=False)
# 上位20個を表示
print(correlations.head(20))

In [ ]:
# 学習データ多重共線性への対処
numeric_train.drop(columns=['SalePrice']).corr()
# GarageCarsとGarageArea：0.882475　1stFlrSFとTotalBsmtSF：0.819530　GrLivAreaとTotRmsAbvGrd：0.825489
# →GarageCarsと1stFlrSFとTotRmsAbvGrdの3つを使用しない

In [ ]:
# ①学習データ 
# 除外したい列をリストで指定
exclude_cols = ['GarageCars', '1stFlrSF', 'TotRmsAbvGrd']
# 相関係数が0.4以上の列名を抽出（SalePrice自身も含まれる）
high_corr_cols = correlations[correlations >= 0.4].index.tolist()
# 除外対象を取り除く
selected_cols = [col for col in high_corr_cols if col not in exclude_cols]
# その列だけを numeric_train に残す
numeric_train = numeric_train[selected_cols]

# ②テストデータ 
# 'SalePrice' を除外した selected_cols を作成
selected_cols_for_test = [col for col in selected_cols if col != 'SalePrice']
# その列だけを numeric_train に残す
numeric_test = numeric_test[selected_cols_for_test]

In [ ]:
# SalePriceとの相関係数を計算し、上位9つの変数を抽出（SalePrice自身は除く）
top_corr = numeric_train.corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False).head(9)

# 散布図を3列×3行で描画
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.flatten()

for i, col in enumerate(top_corr.index):
    sns.scatterplot(x=numeric_train[col], y=numeric_train['SalePrice'], ax=axes[i])
    axes[i].set_title(f'{col} vs SalePrice')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('SalePrice')

plt.tight_layout()
plt.show()


In [ ]:
# # 各特徴量ごとにSalePriceの平均値と標準偏差を表示
# for feature in non_numeric_train:
#     print(f"\nFeature: {feature}")
#     print(train.groupby(feature)['SalePrice'].agg(['mean', 'std']).sort_values(by='mean', ascending=False))

In [ ]:
# グラフの行数と列数を決定（横3列）
n_cols = 3
n_rows = math.ceil(len(non_numeric_train) / n_cols)

# サブプロットを作成
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()  # 1次元に変換して扱いやすくする

# 各特徴量ごとにグラフを描画
for i, feature in enumerate(non_numeric_train):
    stats = train.groupby(feature)['SalePrice'].agg(['mean', 'std']).sort_values(by='mean', ascending=False)
    stats['mean'].plot(kind='bar', yerr=stats['std'], capsize=3, ax=axes[i])
    axes[i].set_title(f'{feature} vs SalePrice')
    axes[i].set_ylabel('Average SalePrice')
    axes[i].tick_params(axis='x', rotation=45)

# 余ったサブプロットを非表示にする
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
MSZoning_map = {
    'A': 0,   # 農業地域（最も低い）
    'I': 1,   # 工業地域
    'C': 2,   # 商業地域
    'RH': 3,  # 高密度住宅地域
    'RM': 4,  # 中密度住宅地域
    'RL': 5,  # 低密度住宅地域
    'RP': 6,  # 公園付き低密度住宅地域
    'FV': 7   # 浮遊住宅地域（最も高い）
}
street_map = {
    'Grvl': 0,  # 砂利道
    'Pave': 1   # 舗装道路
}
alley_map = {
    'NA': 0,    # 裏路地なし
    'Grvl': 1,  # 砂利道
    'Pave': 2   # 舗装
}
lotshape_map = {
    'Reg': 3,   # 整形地（最も高評価）
    'IR1': 2,   # やや不整形
    'IR2': 1,   # 不整形
    'IR3': 0    # 非常に不整形（最も低評価）
}
landcontour_map = {
    'Lvl': 3,   # 平坦（最も高評価）
    'HLS': 2,   # 丘陵（中間）
    'Bnk': 1,   # 傾斜（やや低評価）
    'Low': 0    # 窪地（最も低評価）
}
utilities_map = {
    'ELO': 0,       # 電気のみ（最低評価）
    'NoSeWa': 1,    # 電気・ガスのみ
    'NoSewr': 2,    # 浄化槽（下水なし）
    'AllPub': 3     # 全公共設備（最高評価）
}
lotconfig_map = {
    'Inside': 0,    # 角地でない（標準）
    'FR2': 1,       # 2面接道
    'Corner': 2,    # 角地
    'FR3': 3,       # 3面接道（利便性高）
    'CulDSac': 4    # 袋小路（静かで人気）
}
landslope_map = {
    'Sev': 0,   # 急傾斜（最も低評価）
    'Mod': 1,   # 中程度の傾斜
    'Gtl': 2    # 緩やか（最も高評価）
}
neighborhood_map = {
    'MeadowV': 0, 'IDOTRR': 1, 'BrDale': 2, 'OldTown': 3, 'Edwards': 4,
    'SWISU': 5, 'Blueste': 6, 'NAmes': 7, 'BrkSide': 8, 'Sawyer': 9,
    'NPkVill': 10, 'Mitchel': 11, 'SawyerW': 12, 'NWAmes': 13, 'Gilbert': 14,
    'Blmngtn': 15, 'CollgCr': 16, 'ClearCr': 17, 'Crawfor': 18, 'Somerst': 19,
    'Veenker': 20, 'StoneBr': 21, 'NoRidge': 22, 'NridgHt': 23
}
condition1_map = {
    'RRAn': 0,   # 鉄道に隣接（最も低評価）
    'RRNe': 1,   # 鉄道に近い
    'RRNn': 1,   # 鉄道に近い
    'RRAe': 1,   # 鉄道に近い
    'Artery': 2, # 幹線道路（騒音あり）
    'Feedr': 3,  # フィーダー道路（中間）
    'Norm': 4,   # 通常のアクセス道路（標準）
    'PosA': 5,   # 公園隣接（高評価）
    'PosN': 5    # ポジティブ近隣（高評価）
}
condition2_map = {
    'RRAn': 0,   # 鉄道に隣接（最も低評価）
    'RRAe': 1,   # 鉄道に近い
    'RRNn': 1,   # 鉄道に近い
    'Artery': 2, # 幹線道路（騒音あり）
    'Feedr': 3,  # フィーダー道路（中間）
    'Norm': 4,   # 通常のアクセス道路（標準）
    'PosA': 5,   # 公園隣接（高評価）
    'PosN': 5    # ポジティブ近隣（高評価）
}
bldgtype_map = {
    'Twnhs': 0,     # タウンハウス中央ユニット（最も低評価）
    'TwnhsE': 1,    # タウンハウス端ユニット
    'Duplex': 2,    # 二戸建て住宅
    '2fmCon': 3,    # 二世帯住宅
    '1Fam': 4       # 一戸建て住宅（最も高評価）
}
housestyle_map = {
    '2.5Unf': 0,   # 2.5階建て未完成（最低評価）
    '1.5Unf': 1,   # 1.5階建て未完成
    'SLvl': 2,     # スプリットレベル（複雑構造）
    'Sfoyer': 3,   # スプリット玄関（中間階玄関）
    '1.5Fin': 4,   # 1.5階建て完成
    '1Story': 5,   # 平屋建て（安定人気）
    '2Story': 6,   # 2階建て（高評価）
    '2.5Fin': 7    # 2.5階建て完成（最も高評価）
}
roofstyle_map = {
    'Flat': 0,     # 平らな屋根（排水性や耐久性に課題）
    'Shed': 1,     # 片流れ屋根（モダンだが特殊）
    'Gambrel': 2,  # 納屋風（カントリースタイル）
    'Mansard': 3,  # フランス風（高級感あるが特殊）
    'Gable': 4,    # 三角形の切妻屋根（一般的で人気）
    'Hip': 5       # 寄棟屋根（安定性・耐風性が高く最も高評価）
}
roofmatl_map = {
    'Roll': 0,       # ロール材（最も安価・耐久性低）
    'Tar&Grv': 1,    # タール＆砂利（古い商業用）
    'Membran': 2,    # 防水膜（特殊）
    'WdShngl': 3,    # 木製シングル材（見た目良いが耐久性やや低）
    'WdShake': 4,    # 木製シェイク材（高級感あるがメンテ必要）
    'Metal': 5,      # 金属屋根（耐久性高・モダン）
    'CompShg': 6,    # 合成シングル材（最も一般的・バランス良）
    'ClyTile': 7     # 粘土タイル（高級・耐久性・美観ともに優）
}
exterior_map = {
    'AsphShn': 0,     # アスファルト（最低評価）
    'AsbShng': 1,     # アスベスト（古くて安全性に課題）
    'CBlock': 2,      # コンクリートブロック（無骨・特殊）
    'Other': 3,       # その他（不明・特殊）
    'ImStucc': 4,     # 模造スタッコ（見た目は良いが安価）
    'HdBoard': 5,     # ハードボード（中間）
    'Plywood': 6,     # 合板（中間）
    'Wd Sdng': 7,     # 木製外壁（見た目良いがメンテ必要）
    'Wd Shng': 8,     # 木製シングル（高級感あるが耐久性やや低）
    'Stucco': 9,      # スタッコ（断熱性・美観良好）
    'MetalSd': 10,    # 金属外壁（耐久性高・モダン）
    'Brk Cmn': 11,    # 一般的なレンガ（重厚感あり）
    'BrkFace': 12,    # レンガ調外壁（高級感あり）
    'CmentBd': 13,    # セメント板（耐久性・断熱性良好）
    'VinylSd': 14,    # ビニール外壁（最も人気・メンテ容易）
    'Stone': 15       # 石材（最も高級・耐久性抜群）
}
masvnrtype_map = {
    'NA': 0,        # データなし（最低評価）
    'None': 1,      # 装飾レンガなし
    'BrkCmn': 2,    # 一般的なレンガ（重厚感あり）
    'BrkFace': 3,   # レンガ調の外装（高級感あり）
    'Stone': 4      # 石材（最も高級・耐久性抜群）
}
exterqual_map = {
    'Fa': 0,    # やや劣る（最低評価）
    'TA': 1,    # 平均的（標準）
    'Gd': 2,    # 良好（高評価）
    'Ex': 3     # 最高品質（最も高評価）
}
extercond_map = {
    'Po': 0,    # 劣化が激しい（最低評価）
    'Fa': 1,    # やや劣る
    'TA': 2,    # 平均的（標準）
    'Gd': 3,    # 良好（高評価）
    'Ex': 4     # 最高品質（最も高評価）
}
foundation_map = {
    'Wood': 0,      # 木製基礎（耐久性に劣る・最低評価）
    'Stone': 1,     # 石（古くて特殊）
    'Slab': 2,      # スラブ基礎（安価・中間評価）
    'BrkTil': 3,    # レンガとタイル（見た目良いがやや古い）
    'Cblock': 4,    # コンクリートブロック（一般的・安定）
    'Pconc': 5      # 打設コンクリート（最も高評価・耐久性抜群）
}
bsmtqual_map = {
    'NA': 0,    # データなし（地下室なし、最低評価）
    'Fa': 1,    # やや劣る
    'TA': 2,    # 平均的（標準）
    'Gd': 3,    # 良好（高評価）
    'Ex': 4     # 最高品質（最も高評価）
}
bsmtcond_map = {
    'NA': 0,    # データなし（地下室なし、最低評価）
    'Po': 1,    # 劣化が激しい
    'Fa': 2,    # やや劣る
    'TA': 3,    # 平均的（標準）
    'Gd': 4     # 良好（最も高評価）
}
bsmtexposure_map = {
    'NA': 0,    # 地下室なし（最低評価）
    'No': 1,    # 露出なし
    'Mn': 2,    # 最小限の露出
    'Av': 3,    # 平均的な露出
    'Gd': 4     # 良好な露出（最も高評価）
}
bsmtfintype_map = {
    'NA': 0,     # 地下室なし（最低評価）
    'Unf': 1,    # 仕上げなし
    'LwQ': 2,    # 低品質の居住空間
    'BLQ': 3,    # 下位の居住空間
    'Rec': 4,    # レクリエーションルーム（中間評価）
    'ALQ': 5,    # 平均的な居住空間
    'GLQ': 6     # 高品質の仕上げ（最も高評価）
}
heating_map = {
    'Grav': 0,     # 重力式暖房（旧式・最低評価）
    'Wall': 1,     # 壁付け暖房（局所的・やや旧式）
    'OthW': 2,     # その他の温水・蒸気暖房（特殊）
    'GasW': 3,     # ガス温水または蒸気暖房（中間評価）
    'GasA': 4,     # ガス温風暖房（一般的・効率良）
    'Floor': 5     # 床暖房（快適性・効率性ともに最も高評価）
}
heatingqc_map = {
    'Po': 0,    # 劣る（最低評価）
    'Fa': 1,    # やや劣る
    'TA': 2,    # 平均（標準）
    'Gd': 3,    # 良好（高評価）
    'Ex': 4     # 優秀（最も高評価）
}
centralair_map = {
    'N': 0,    # なし（最低評価）
    'Y': 1     # あり（高評価）
}
electrical_map = {
    'NA': 0,      # 電気設備なし（最低評価）
    'FuseP': 1,   # 安全性に懸念あり（旧式）
    'FuseF': 2,   # やや古い設備
    'FuseA': 3,   # 平均的な安全性
    'Mix': 4,     # 複数の配線方式が混在（やや特殊）
    'SBrkr': 5    # 標準ブレーカー（最も一般的・高評価）
}
kitchenqual_map = {
    'Fa': 0,    # やや劣る（最低評価）
    'TA': 1,    # 平均（標準）
    'Gd': 2,    # 良好（高評価）
    'Ex': 3     # 優秀（最も高評価）
}
functional_map = {
    'Sev': 0,     # 深刻な欠陥（最低評価）
    'Maj2': 1,    # 重大な欠陥（大）
    'Maj1': 2,    # 重大な欠陥（小）
    'Mod': 3,     # 中程度の欠陥
    'Min2': 4,    # 軽微な欠陥（やや多い）
    'Min1': 5,    # 軽微な欠陥（少ない）
    'Typ': 6      # 標準的な機能性（最も高評価）
}
fireplacequ_map = {
    'NA': 0,    # 暖炉なし（最低評価）
    'Po': 1,    # 劣る
    'Fa': 2,    # やや劣る
    'TA': 3,    # 平均
    'Gd': 4,    # 良好
    'Ex': 5     # 優秀（最も高評価）
}
garagetype_map = {
    'NA': 0,        # ガレージなし（最低評価）
    'CarPort': 1,   # 簡易型（屋根のみ）
    'Detchd': 2,    # 独立型（利便性やや低）
    'Basment': 3,   # 地下ガレージ（特殊・やや高評価）
    'Attchd': 4,    # 住宅に接続（一般的・高評価）
    'BuiltIn': 5,   # 建物一体型（最も高評価）
    '2Types': 6     # 複数タイプ（柔軟性あり・高評価）
}
garagefinish_map = {
    'NA': 0,    # ガレージなし（最低評価）
    'Unf': 1,   # 未仕上げ
    'RFn': 2,   # ラフ仕上げ（中間評価）
    'Fin': 3    # 仕上げ済み（最も高評価）
}
garagequal_map = {
    'NA': 0,    # ガレージなし（最低評価）
    'Po': 1,    # 劣る
    'Fa': 2,    # やや劣る
    'TA': 3,    # 平均
    'Gd': 4,    # 良好
    'Ex': 5     # 優秀（最も高評価）
}
garagecond_map = {
    'NA': 0,    # ガレージなし（最低評価）
    'Po': 1,    # 劣る
    'Fa': 2,    # やや劣る
    'TA': 3,    # 平均
    'Gd': 4,    # 良好
    'Ex': 5     # 優秀（最も高評価）
}
paveddrive_map = {
    'N': 0,    # 砂利（最低評価）
    'P': 1,    # 部分舗装（中間評価）
    'Y': 2     # 完全舗装（最も高評価）
}
poolqc_map = {
    'NA': 0,    # プールなし（最低評価）
    'Fa': 1,    # やや劣る
    'Gd': 2,    # 良好
    'Ex': 3     # 優秀（最も高評価）
}
fence_map = {
    'NA': 0,       # フェンスなし（最低評価）
    'MnWw': 1,     # 最低限の木製またはワイヤーフェンス
    'MnPrv': 2,    # 最低限のプライバシーフェンス
    'GdWo': 3,     # 高品質な木製フェンス
    'GdPrv': 4     # 高品質なプライバシーフェンス（最も高評価）
}
miscfeature_map = {
    'NA': 0,     # その他設備なし（最低評価）
    'Othr': 1,   # その他（内容不明・低評価）
    'Shed': 2,   # 収納小屋（実用的）
    'Gar2': 3,   # 2つ目のガレージ（利便性・高評価）
    'TenC': 4    # テニスコート（希少・最も高評価）
}
saletype_map = {
    'Oth': 0,      # その他（内容不明・最低評価）
    'COD': 1,      # 裁判所・遺産売買（特殊・不安定）
    'ConLw': 2,    # 頭金・金利ともに低い契約
    'ConLD': 3,    # 頭金が低い契約
    'ConLI': 4,    # 金利が低い契約
    'Con': 5,      # 通常の契約売買
    'CWD': 6,      # 保証付き現金売買
    'WD': 7,       # 保証付き通常売買（安定・高評価）
    'New': 8       # 新築（最も高評価）
}
salecondition_map = {
    'Partial': 0,    # 未完成住宅（価格不安定・最低評価）
    'Abnorml': 1,    # 異常売買（特殊・不安定）
    'AdjLand': 2,    # 隣接地購入（特殊）
    'Alloca': 3,     # 分割売買（やや特殊）
    'Family': 4,     # 家族間売買（価格が市場と乖離しやすい）
    'Normal': 5      # 通常売買（最も安定・高評価）
}


In [ ]:
train['MSZoning'] = train['MSZoning'].map(MSZoning_map).fillna(0)
test['MSZoning'] = test['MSZoning'].map(MSZoning_map).fillna(0)

train['Street'] = train['Street'].map(street_map).fillna(0)
test['Street'] = test['Street'].map(street_map).fillna(0)

train['Alley'] = train['Alley'].map(alley_map).fillna(0)
test['Alley'] = test['Alley'].map(alley_map).fillna(0)

train['LotShape'] = train['LotShape'].map(lotshape_map).fillna(0)
test['LotShape'] = test['LotShape'].map(lotshape_map).fillna(0)

train['LandContour'] = train['LandContour'].map(landcontour_map).fillna(0)
test['LandContour'] = test['LandContour'].map(landcontour_map).fillna(0)

train['Utilities'] = train['Utilities'].map(utilities_map).fillna(0)
test['Utilities'] = test['Utilities'].map(utilities_map).fillna(0)

train['LotConfig'] = train['LotConfig'].map(lotconfig_map).fillna(0)
test['LotConfig'] = test['LotConfig'].map(lotconfig_map).fillna(0)

train['LandSlope'] = train['LandSlope'].map(landslope_map).fillna(0)
test['LandSlope'] = test['LandSlope'].map(landslope_map).fillna(0)

train['Neighborhood'] = train['Neighborhood'].map(neighborhood_map).fillna(0)
test['Neighborhood'] = test['Neighborhood'].map(neighborhood_map).fillna(0)

train['Condition1'] = train['Condition1'].map(condition1_map).fillna(0)
test['Condition1'] = test['Condition1'].map(condition1_map).fillna(0)

train['Condition2'] = train['Condition2'].map(condition2_map).fillna(0)
test['Condition2'] = test['Condition2'].map(condition2_map).fillna(0)

train['BldgType'] = train['BldgType'].map(bldgtype_map).fillna(0)
test['BldgType'] = test['BldgType'].map(bldgtype_map).fillna(0)

train['HouseStyle'] = train['HouseStyle'].map(housestyle_map).fillna(0)
test['HouseStyle'] = test['HouseStyle'].map(housestyle_map).fillna(0)

train['RoofStyle'] = train['RoofStyle'].map(roofstyle_map).fillna(0)
test['RoofStyle'] = test['RoofStyle'].map(roofstyle_map).fillna(0)

train['RoofMatl'] = train['RoofMatl'].map(roofmatl_map).fillna(0)
test['RoofMatl'] = test['RoofMatl'].map(roofmatl_map).fillna(0)

train['Exterior1st'] = train['Exterior1st'].map(exterior_map).fillna(0)
test['Exterior1st'] = test['Exterior1st'].map(exterior_map).fillna(0)

train['Exterior2nd'] = train['Exterior2nd'].map(exterior_map).fillna(0)
test['Exterior2nd'] = test['Exterior2nd'].map(exterior_map).fillna(0)

train['MasVnrType'] = train['MasVnrType'].map(masvnrtype_map).fillna(0)
test['MasVnrType'] = test['MasVnrType'].map(masvnrtype_map).fillna(0)

train['ExterQual'] = train['ExterQual'].map(exterqual_map).fillna(0)
test['ExterQual'] = test['ExterQual'].map(exterqual_map).fillna(0)

train['ExterCond'] = train['ExterCond'].map(extercond_map).fillna(0)
test['ExterCond'] = test['ExterCond'].map(extercond_map).fillna(0)

train['Foundation'] = train['Foundation'].map(foundation_map).fillna(0)
test['Foundation'] = test['Foundation'].map(foundation_map).fillna(0)

train['BsmtQual'] = train['BsmtQual'].map(bsmtqual_map).fillna(0)
test['BsmtQual'] = test['BsmtQual'].map(bsmtqual_map).fillna(0)

train['BsmtCond'] = train['BsmtCond'].map(bsmtcond_map).fillna(0)
test['BsmtCond'] = test['BsmtCond'].map(bsmtcond_map).fillna(0)

train['BsmtExposure'] = train['BsmtExposure'].map(bsmtexposure_map).fillna(0)
test['BsmtExposure'] = test['BsmtExposure'].map(bsmtexposure_map).fillna(0)

train['BsmtFinType1'] = train['BsmtFinType1'].map(bsmtfintype_map).fillna(0)
test['BsmtFinType1'] = test['BsmtFinType1'].map(bsmtfintype_map).fillna(0)

train['BsmtFinType2'] = train['BsmtFinType2'].map(bsmtfintype_map).fillna(0)
test['BsmtFinType2'] = test['BsmtFinType2'].map(bsmtfintype_map).fillna(0)

train['Heating'] = train['Heating'].map(heating_map).fillna(0)
test['Heating'] = test['Heating'].map(heating_map).fillna(0)

train['HeatingQC'] = train['HeatingQC'].map(heatingqc_map).fillna(0)
test['HeatingQC'] = test['HeatingQC'].map(heatingqc_map).fillna(0)

train['CentralAir'] = train['CentralAir'].map(centralair_map).fillna(0)
test['CentralAir'] = test['CentralAir'].map(centralair_map).fillna(0)

train['Electrical'] = train['Electrical'].map(electrical_map).fillna(0)
test['Electrical'] = test['Electrical'].map(electrical_map).fillna(0)

train['KitchenQual'] = train['KitchenQual'].map(kitchenqual_map).fillna(0)
test['KitchenQual'] = test['KitchenQual'].map(kitchenqual_map).fillna(0)

train['Functional'] = train['Functional'].map(functional_map).fillna(0)
test['Functional'] = test['Functional'].map(functional_map).fillna(0)

train['FireplaceQu'] = train['FireplaceQu'].map(fireplacequ_map).fillna(0)
test['FireplaceQu'] = test['FireplaceQu'].map(fireplacequ_map).fillna(0)

train['GarageType'] = train['GarageType'].map(garagetype_map).fillna(0)
test['GarageType'] = test['GarageType'].map(garagetype_map).fillna(0)

train['GarageFinish'] = train['GarageFinish'].map(garagefinish_map).fillna(0)
test['GarageFinish'] = test['GarageFinish'].map(garagefinish_map).fillna(0)

train['GarageQual'] = train['GarageQual'].map(garagequal_map).fillna(0)
test['GarageQual'] = test['GarageQual'].map(garagequal_map).fillna(0)

train['GarageCond'] = train['GarageCond'].map(garagecond_map).fillna(0)
test['GarageCond'] = test['GarageCond'].map(garagecond_map).fillna(0)

train['PavedDrive'] = train['PavedDrive'].map(paveddrive_map).fillna(0)
test['PavedDrive'] = test['PavedDrive'].map(paveddrive_map).fillna(0)

train['PoolQC'] = train['PoolQC'].map(poolqc_map).fillna(0)
test['PoolQC'] = test['PoolQC'].map(poolqc_map).fillna(0)

train['Fence'] = train['Fence'].map(fence_map).fillna(0)
test['Fence'] = test['Fence'].map(fence_map).fillna(0)

train['MiscFeature'] = train['MiscFeature'].map(miscfeature_map).fillna(0)
test['MiscFeature'] = test['MiscFeature'].map(miscfeature_map).fillna(0)

train['SaleType'] = train['SaleType'].map(saletype_map).fillna(0)
test['SaleType'] = test['SaleType'].map(saletype_map).fillna(0)

train['SaleCondition'] = train['SaleCondition'].map(salecondition_map).fillna(0)
test['SaleCondition'] = test['SaleCondition'].map(salecondition_map).fillna(0)

In [ ]:
# for col in non_numeric_train:
#     # 各カテゴリの平均SalePriceを計算
#     mean_map = train.groupby(col)['SalePrice'].mean().sort_values().reset_index()
#     mean_map['rank'] = range(len(mean_map))  # 平均値が低い順に0,1,2...

#     # マッピング辞書を作成
#     rank_map = dict(zip(mean_map[col], mean_map['rank']))

#     # 学習データの元の列を置き換え
#     train[col] = train[col].map(rank_map)

#     # テストデータの元の列も置き換え（未知カテゴリは -1）
#     test[col] = test[col].map(rank_map).fillna(-1)



In [ ]:
# 数値型の列だけを抽出
train = train.select_dtypes(include=['number'])

# 相関係数を計算し、SalePriceとの相関が高い上位16を抽出（SalePrice自身は除く）
top_corr = train.corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False).head(16)

# 散布図を4列×4行で描画
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(top_corr.index):
    sns.scatterplot(x=train[col], y=train['SalePrice'], ax=axes[i])
    axes[i].set_title(f'{col} vs SalePrice')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('SalePrice')

plt.tight_layout()
plt.show()

In [ ]:
# 外れ値の除去
train = train[train['GrLivArea'] <= 4500]
# train = train[train['GarageCars'] < 4.0]
# train = train[train['GarageArea'] <= 1200]
train = train[train['TotalBsmtSF'] <= 4000]
train = train[train['1stFlrSF'] <= 4000]


In [ ]:
# 数値型の列だけを抽出
train = train.select_dtypes(include=['number'])

# 相関係数を計算し、SalePriceとの相関が高い上位12を抽出（SalePrice自身は除く）
top_corr = train.corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False).head(16)

# 散布図を4列×4行で描画
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(top_corr.index):

    # 散布図を描画
    sns.scatterplot(x=train[col], y=train['SalePrice'], ax=axes[i])
    axes[i].set_title(f'{col} vs SalePrice')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('SalePrice')

plt.tight_layout()
plt.show()

In [ ]:
top20 = train.corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False).head(20).index

# 12特徴量の相関行列（絶対値）
corr_matrix = train[top20].corr().abs()

# 上三角行列だけを対象にする（自己相関を除く）
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# 相関係数が0.7以上のペアを抽出
high_corr_pairs = [(col, row, upper.loc[row, col])
                   for col in upper.columns
                   for row in upper.index
                   if upper.loc[row, col] >= 0.7]

# 結果をDataFrameで表示
import pandas as pd
high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature 1', 'Feature 2', 'Correlation'])
print(high_corr_df.sort_values(by='Correlation', ascending=False))


In [ ]:
# ①学習データ 
# 除外したい列をリストで指定
exclude_cols = ['GarageArea','TotRmsAbvGrd','Fireplaces','TotalBsmtSF','GarageYrBlt']
# 相関係数が0.4以上の列名を抽出（SalePrice自身も含まれる）
high_corr_cols = correlations[correlations >= 0.4].index.tolist()
# 除外対象を取り除く
selected_cols = [col for col in high_corr_cols if col not in exclude_cols]
# その列だけを train に残す
train = train[selected_cols]

# ②テストデータ 
# 'SalePrice' を除外した selected_cols を作成
selected_cols_for_test = [col for col in selected_cols if col != 'SalePrice']
# その列だけを numeric_train に残す
test = test[selected_cols_for_test]

In [ ]:
# 数値型の列だけを抽出
train = train.select_dtypes(include=['number'])

# 相関係数を計算し、SalePriceとの相関が高い上位12を抽出（SalePrice自身は除く）
top_corr = train.corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False).head(12)

# 散布図を4列×4行で描画
fig, axes = plt.subplots(4, 3, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(top_corr.index):

    sns.scatterplot(x=train[col], y=train['SalePrice'], ax=axes[i])
    axes[i].set_title(f'{col} vs SalePrice')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('SalePrice')

plt.tight_layout()
plt.show()

In [ ]:
# 相関上位12の特徴量（SalePriceは除いてある）
selected_features = top_corr.index.tolist()  # 11個の特徴量名
# テストデータから同じ特徴量だけを抽出
test_selected = test[selected_features]
print("テストデータの特徴量数:", test_selected.shape[1])

In [ ]:
# 特徴量とラベルに分ける
X = train.drop('SalePrice', axis=1).values  # 入力データ(数値データのみ、カテゴリ変数は扱わない)
y = train['SalePrice'].values               # 出力データ

In [ ]:
# 学習データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [ ]:
feature_names = [f'X{i}' for i in range(X.shape[1])]

In [ ]:
# StandardScalerクラスのインスタンス化
scaler = StandardScaler()
# 標準化の変換モデルの生成
scaler.fit(X_train)
# 標準化の変換モデルを利用したスケールの変換
X_train_scaled = scaler.transform(X_train)

# NumPy配列 → pandas.DataFrame に変換
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names)
# 統計情報を表示
X_train_scaled_df.describe()

In [ ]:
# StandardScalerクラスのインスタンス化
# scaler = StandardScaler()
# 標準化の変換モデルの生成 ★標準化の変換モデルの生成に、説明変数のテストデータを利用しない
# scaler.fit(X_test)　★標準化の変換モデルの生成に、説明変数のテストデータを利用しない
# 標準化の変換モデルを利用したスケールの変換
X_test_scaled = scaler.transform(X_test)

# NumPy配列 → pandas.DataFrame に変換
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_names)
# 統計情報を表示
X_test_scaled_df.describe()

In [ ]:
# 特徴量とラベルの学習・テストデータの形状を表示
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# 線形回帰モデルのインスタンスを作成
model = LinearRegression()

In [ ]:
# 学習用の特徴量とラベルを使って線形回帰モデルを訓練する
# model.fit(X_train, y_train)
model.fit(X_train_scaled, y_train)

In [ ]:
# 学習データに対する決定係数（R²スコア）を計算し、モデルの説明力を評価する
r2_train = round(model.score(X_train_scaled, y_train), 5)
print("R²スコア（学習データ）:", r2_train)

In [ ]:
# テストデータに対する決定係数（R²スコア）を計算し、汎化性能を評価する
r2_test = round(model.score(X_test_scaled, y_test), 5)
print("R²スコア（テストデータ）:", r2_test)

In [ ]:
# テストデータの予測結果を実測値と比較し、予測精度を散布図で描画する
y_pred = model.predict(X_test_scaled)

plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel('実際の価格')
plt.ylabel('予測された価格')
plt.title('住宅価格の予測精度（実測 vs 予測）')
plt.grid(True)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # 完全一致ライン
plt.show()


In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8,6))
sns.histplot(residuals, kde=True)
plt.title('残差（予測誤差）の分布')
plt.xlabel('残差')
plt.grid(True)
plt.show()


In [ ]:
# 学習と同じ前処理を行う
# スケーリング（学習時の scaler を使う）
test_scaled = scaler.transform(test.values)
# NumPy配列 → pandas.DataFrame に変換
test_scaled_df = pd.DataFrame(test_scaled, columns=feature_names)

# 予測
y_pred = model.predict(test_scaled)

# 提出用DataFrame
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': y_pred.flatten()
})

# CSV保存
submission.to_csv('LinearRegression.csv', index=False)
